# Pyramid construction — per coarsening factor, per build mode

Fixed `N = 1_000_000`.  For point clouds and graphs (`N` nodes + ~3N/2
edges) we sweep coarsening ratio `R ∈ {2, 4, 8}` over three pyramid
levels (resulting pyramids span level 0 / 1 / 2 / 3) and three
**cross-level link storage modes**:

| `cross_level_storage` | meaning |
| --- | --- |
| `"none"`     | no cross-resolution links |
| `"implicit"` | cross-chunk links only across resolutions (one direction: finer → coarser, `+Δ`) |
| `"explicit"` | full multires links: both `+Δ` (finer level) and `-Δ` (coarser level) |

Per `(geom, R, mode, level)` we collect: total build time, vertex
count, total link rows on disk (`cross_chunk_links/<Δ>/data` +
`links/<Δ>/<chunk_key>` across every Δ), per-level disk size, and
read-all-vertices time.

Build timing is averaged over `N_BUILD_RUNS = 5`; read timing over
`N_BUILD_RUNS × N_READ_RUNS = 15` samples (Student's t 95 % CI).
Vertex counts, link counts and disk sizes are deterministic given
the fixed seed, so they are captured once on `run == 0`.

Runtime: ~15–25 minutes on a laptop.

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'


N_RUNS = 10
T95_DF9 = 2.262  # scipy.stats.t.ppf(0.975, df=9) — hard-coded to avoid scipy dep


def _mean_ci95(samples):
    """(mean, half-width) for a 1-D sample using Student's t, df=n-1."""
    arr = np.asarray(samples, dtype=float)
    if arr.size < 2:
        return float(arr.mean()) if arr.size else 0.0, 0.0
    m = arr.mean()
    s = arr.std(ddof=1)
    hw = T95_DF9 * s / np.sqrt(arr.size)
    return float(m), float(hw)

## 1 · Setup

In [ ]:
import zarr
from zarr_vectors.types.points  import write_points, read_points
from zarr_vectors.types.graphs  import write_graph,  read_graph
from zarr_vectors.multiresolution.coarsen import build_pyramid
from zarr_vectors.core.store    import open_store, read_level_metadata

N       = 1_000_000
RATIOS  = [2, 4, 8]
MODES   = ['none', 'implicit', 'explicit']
N_LVLS  = 3                   # builds levels 1, 2, 3 from level 0
CHUNK   = (200.0, 200.0, 200.0)
BIN     = (50.0,  50.0,  50.0)
SEED    = 0
N_BUILD_RUNS = 5
N_READ_RUNS  = 3


def _level_link_count(store_path, level):
    """Total link rows on disk at a pyramid level.

    Sums rows across every ``Δ`` and every chunk:
      - ``<level>/cross_chunk_links/<Δ>/data`` — one array per Δ
      - ``<level>/links/<Δ>/<chunk_key>``    — one array per chunk per Δ
    """
    grp = zarr.open_group(str(store_path), path=str(level), mode='r')
    total = 0
    for family in ('cross_chunk_links', 'links'):
        if family not in grp:
            continue
        for delta_seg in grp[family]:
            sub = grp[family][delta_seg]
            for name in sub:
                arr = sub[name]
                try:
                    total += int(arr.shape[0])
                except Exception:
                    continue
    return total

## 2 · Run the sweep

In [ ]:
rng = np.random.default_rng(SEED)
positions = rng.uniform(0, 1000, (N, 3)).astype(np.float32)
# ~3N/2 random undirected edges, dedup'd against self-loops.
src = rng.integers(0, N, size=3 * N // 2)
dst = rng.integers(0, N, size=3 * N // 2)
mask = src != dst
edges = np.stack([src[mask], dst[mask]], axis=1).astype(np.int64)

GEOMETRIES = [
    ('points', write_points, (positions,),         read_points),
    ('graph',  write_graph,  (positions, edges),   read_graph),
]

# (geom, ratio, mode) -> raw measurements.
data = {}

for ratio in RATIOS:
    for geom, write_fn, args, read_fn in GEOMETRIES:
        for mode in MODES:
            build_times = np.zeros(N_BUILD_RUNS)
            read_times  = {k: [] for k in range(N_LVLS + 1)}
            per_level   = None              # filled on run == 0
            for run in range(N_BUILD_RUNS):
                store_path = _new_store(
                    f'pyr_{geom}_r{ratio}_{mode}_run{run}'
                )
                write_fn(store_path, *args,
                         chunk_shape=CHUNK, bin_shape=BIN)
                t_build, _ = _time(
                    build_pyramid, store_path,
                    factors=[(float(ratio), 1.0)] * N_LVLS,
                    cross_level_storage=mode,
                )
                build_times[run] = t_build

                if run == 0:
                    root = open_store(str(store_path), mode='r')
                    per_level = []
                    for k in range(N_LVLS + 1):
                        lm = read_level_metadata(root, k)
                        per_level.append({
                            'vertex_count': int(lm.vertex_count),
                            'links_total': _level_link_count(store_path, k),
                            'disk_bytes':  _store_bytes(Path(store_path) / str(k)),
                        })

                for k in range(N_LVLS + 1):
                    for _ in range(N_READ_RUNS):
                        t_r, _ = _time(read_fn, store_path, level=k)
                        read_times[k].append(t_r)

                shutil.rmtree(Path(store_path).parent, ignore_errors=True)
            data[(geom, ratio, mode)] = {
                'build_times': build_times,
                'read_times':  {k: np.array(v) for k, v in read_times.items()},
                'per_level':   per_level,
            }

# Tidy dataframe: one row per (geom, ratio, mode, level).
rows = []
for (geom, ratio, mode), d in data.items():
    t_mean, t_hw = _mean_ci95(d['build_times'])
    for k in range(N_LVLS + 1):
        r_mean, r_hw = _mean_ci95(d['read_times'][k])
        rows.append({
            'geom':  geom,
            'ratio': ratio,
            'mode':  mode,
            'level': k,
            'build_total_s_mean': round(t_mean, 4),
            'build_total_s_hw':   round(t_hw,   4),
            'vertex_count': d['per_level'][k]['vertex_count'],
            'links_total':  d['per_level'][k]['links_total'],
            'disk_MB':      round(d['per_level'][k]['disk_bytes'] / 1e6, 3),
            'read_all_s_mean': round(r_mean, 4),
            'read_all_s_hw':   round(r_hw,   4),
        })
df = pd.DataFrame(rows)

## 3 · Results — full table

In [ ]:
df

## 4 · Links per level (the headline finding)

Total link rows on disk per `(level, mode)` at `R = 4`.  `none` writes
only intra-level cross-chunk links; `implicit` adds the `+Δ`
cross-level arrays at the finer level; `explicit` mirrors those as
`-Δ` arrays at the coarser level too.

In [ ]:
pivot = (df[df.ratio == 4]
         .pivot_table(index=['geom', 'level'],
                      columns='mode',
                      values='links_total',
                      sort=False)[MODES])
pivot

## 5 · Plot

In [ ]:
MODE_COLORS = {'none': 'C0', 'implicit': 'C1', 'explicit': 'C2'}
GEOMS = ['points', 'graph']
PIVOT_R = 4   # the ratio used in the per-level panels

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for row, geom in enumerate(GEOMS):
    # --- Col 0: build time vs R, grouped by mode --------------------
    ax = axes[row, 0]
    x_pos = np.arange(len(RATIOS))
    for j, mode in enumerate(MODES):
        means, hws = [], []
        for r in RATIOS:
            sub = df[(df.geom == geom)
                     & (df.ratio == r)
                     & (df.mode == mode)].iloc[0]
            means.append(sub['build_total_s_mean'])
            hws.append(sub['build_total_s_hw'])
        ax.bar(
            x_pos + (j - 1) * 0.27, means, width=0.27,
            yerr=hws, capsize=3,
            color=MODE_COLORS[mode], label=mode,
        )
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'R={r}' for r in RATIOS])
    ax.set_ylabel('build time (s)')
    ax.set_title(f'{geom}: total build time')
    ax.grid(True, axis='y', alpha=0.3)

    # --- Col 1: per-level disk size @ R=PIVOT_R ---------------------
    ax = axes[row, 1]
    x_pos = np.arange(N_LVLS + 1)
    for j, mode in enumerate(MODES):
        means = [
            df[(df.geom == geom) & (df.ratio == PIVOT_R)
               & (df.mode == mode) & (df.level == k)].iloc[0]['disk_MB']
            for k in range(N_LVLS + 1)
        ]
        ax.bar(
            x_pos + (j - 1) * 0.27, means, width=0.27,
            color=MODE_COLORS[mode], label=mode,
        )
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'L{k}' for k in range(N_LVLS + 1)])
    ax.set_ylabel('disk size (MB)')
    ax.set_title(f'{geom}: disk per level @ R={PIVOT_R}')
    ax.grid(True, axis='y', alpha=0.3)

    # --- Col 2: per-level read time @ R=PIVOT_R ---------------------
    ax = axes[row, 2]
    for j, mode in enumerate(MODES):
        means, hws = [], []
        for k in range(N_LVLS + 1):
            sub = df[(df.geom == geom) & (df.ratio == PIVOT_R)
                     & (df.mode == mode) & (df.level == k)].iloc[0]
            means.append(sub['read_all_s_mean'])
            hws.append(sub['read_all_s_hw'])
        ax.bar(
            x_pos + (j - 1) * 0.27, means, width=0.27,
            yerr=hws, capsize=3,
            color=MODE_COLORS[mode], label=mode,
        )
    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'L{k}' for k in range(N_LVLS + 1)])
    ax.set_ylabel('read time (s)')
    ax.set_title(f'{geom}: read per level @ R={PIVOT_R}')
    ax.grid(True, axis='y', alpha=0.3)

axes[0, 0].legend(loc='best', fontsize=9)
fig.suptitle(
    f'Pyramid build vs cross-level link mode '
    f'(N={N:,}, {N_BUILD_RUNS} build runs, 95% CI)',
)
plt.tight_layout()